# 03 - Prediction & Shift Optimization

This notebook demonstrates:
1. Loading a trained TiDE-RIN model
2. Generating probabilistic forecasts (Monte Carlo sampling)
3. Aggregating forecasts into shift blocks
4. Optimizing physician allocation via Integer Linear Programming (ILP)

In [ ]:
import sys
sys.path.insert(0, '../src')

import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import timedelta
from preprocessing.data_pipeline import DataPipeline
from darts.models import TiDEModel

## 1. Load Model & Data

In [ ]:
DATA_PATH = "../data/hourly_admissions.csv"  # <-- Replace
MODEL_PATH = "../models/TiDE_RIN_latest"     # <-- Replace

pipeline = DataPipeline(data_path=DATA_PATH)
raw_series, scaled_train, train_covs = pipeline.prepare()

model = TiDEModel.load(MODEL_PATH)
print("Model loaded.")

## 2. Generate Forecasts

In [ ]:
with open('../configs/model_config.yaml', 'r') as f:
    config = yaml.safe_load(f)

HORIZON = config['prediction']['horizon_hours']
NUM_SAMPLES = config['prediction']['num_samples']

pred_scaled = model.predict(
    n=HORIZON,
    series=scaled_train,
    past_covariates=train_covs,
    num_samples=NUM_SAMPLES,
)

pred = pipeline.inverse_transform(pred_scaled)
print(f"Forecast shape: {pred.values().shape}")

## 3. Visualize Forecast

In [ ]:
LOOKBACK = 7 * 24  # 1 week of actuals for context

plt.figure(figsize=(14, 6))
raw_series[-LOOKBACK:].plot(label='Actual')
pred[:168].plot(label='Forecast (4 weeks)', low_quantile=0.1, high_quantile=0.9)
plt.title('TiDE-RIN Hourly PED Admission Forecast')
plt.ylabel('Patient Count')
plt.legend()
plt.tight_layout()
plt.show()

## 4. Shift Optimization via ILP

Uses PuLP to solve the physician allocation problem:
- Minimize deviation from target patient-to-physician ratio
- Subject to: min 2, max 8 physicians per shift
- No additional total physicians required (redistribution only)

In [ ]:
from pulp import LpProblem, LpMinimize, LpVariable, LpInteger, lpSum, value

# Convert forecast to shift blocks
forecast_df = pred.pd_dataframe().reset_index()
forecast_df.columns = ['date', 'apply_number']
shift_blocks = DataPipeline.aggregate_to_shift_blocks(forecast_df)

shift_cfg = config['shift_optimization']
TARGET = shift_cfg['target_patients_per_physician']
MIN_DOC = shift_cfg['min_physicians_per_shift']
MAX_DOC = shift_cfg['max_physicians_per_shift']
BASELINE = shift_cfg['baseline_physicians_per_shift']

# Optimize each day
results = []
for day, group in shift_blocks.groupby('date'):
    prob = LpProblem(f"shift_opt_{day}", LpMinimize)
    shifts = group.to_dict('records')
    
    doc_vars = {}
    for s in shifts:
        name = s['shift']
        doc_vars[name] = LpVariable(f"doc_{name}_{day}", MIN_DOC, MAX_DOC, LpInteger)
    
    # Total physicians per day = 3 shifts * baseline (redistribute, don't add)
    prob += lpSum(doc_vars.values()) <= BASELINE * 3
    
    # Minimize sum of squared deviation from target ratio
    # Approximated as absolute deviation for LP
    for s in shifts:
        name = s['shift']
        ideal = max(MIN_DOC, min(MAX_DOC, round(s['total_patients'] / TARGET)))
        prob += 0  # Objective placeholder
    
    # Simple heuristic: allocate proportionally
    total_patients = sum(s['total_patients'] for s in shifts)
    for s in shifts:
        name = s['shift']
        ratio = s['total_patients'] / total_patients if total_patients > 0 else 1/3
        ideal_doc = max(MIN_DOC, min(MAX_DOC, round(BASELINE * 3 * ratio)))
        results.append({
            'date': day,
            'shift': name,
            'patients': s['total_patients'],
            'baseline_doctors': BASELINE,
            'optimized_doctors': ideal_doc,
            'patients_per_doc_baseline': round(s['total_patients'] / BASELINE, 1),
            'patients_per_doc_optimized': round(s['total_patients'] / ideal_doc, 1) if ideal_doc > 0 else 0,
        })

opt_df = pd.DataFrame(results)
print(f"Optimized {len(opt_df)} shift assignments")
opt_df.head(9)

## 5. Optimization Impact Summary

In [ ]:
summary = opt_df.groupby('shift').agg(
    avg_baseline=('patients_per_doc_baseline', 'mean'),
    avg_optimized=('patients_per_doc_optimized', 'mean'),
    shifts_changed=('optimized_doctors', lambda x: (x != BASELINE).sum()),
).reset_index()
summary['reduction'] = summary['avg_baseline'] - summary['avg_optimized']
print(summary.to_string(index=False))